# IST Landscape Pre-Processing

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import celldega as dega
from glob import glob
import numpy as np
import pandas as pd
import scanpy as sc
import tifffile
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import geopandas as gpd

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.5 when it was built against 1.14.6, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [3]:
dataset_name = 'E12_71'
inst_slice = 'T8'

path_data = 'data/IST_data/Substrate_' + dataset_name + '/'
path_landscape_files = 'data/landscape_files/IST_mouse_cranium/'

In [4]:
image_scale = 1.0
suffix = '.webp[Q=100]'

# Main Function

In [ ]:
sample = 'Substrate_E12_71'
data_dir = 'data/IST_data/' # f'data/xenium_data/'
path_landscape_files=f'data/landscape_files/IST_{sample}_2025-07-29'

tile_size=250
image_tile_layer='h&e'

dega.pre.main(
    sample=sample,
    data_root_dir=data_dir,
    tile_size=tile_size,
    image_tile_layer=image_tile_layer,
    path_landscape_files=path_landscape_files,
    use_int_index=True,
)

Starting preprocessing for sample: Substrate_E12_71
Skipping transform step, using existing data/landscape_files/IST_Substrate_E12_71_2025-07-29/micron_to_image_transform.csv

========Check if all required files or directories exist========
All required files or directories for technology 'IST' are present in 'data/IST_data/Substrate_E12_71'.
Skipping meta cell generation, found data/landscape_files/IST_Substrate_E12_71_2025-07-29/cell_metadata.parquet
IST: read CBG matrix
['cell10000', 'cell10001', 'cell10002', 'cell10003', 'cell10004']
Duplicate columns found in CBG matrix. Making column names unique.
Skipping meta gene file creation, found data/landscape_files/IST_Substrate_E12_71_2025-07-29/meta_gene.parquet
Skipping CBG gene parquets, directory data/landscape_files/IST_Substrate_E12_71_2025-07-29/cbg already populated

======== IST: Image tiles ========

======== IST: Transcript Tiles ========
========Make pseudo transcript tiles========
read path_spot_positions: data/landscape_fi

In [31]:

# tmp.set_index(0, inplace=True)
tmp.index.name = None
tmp.columns = ['x', 'y']
tmp = tmp.astype(float)
# tmp['x'] = (tmp['x'] - gc.loc[inst_slice + '_' + dataset, 'X_shift']) * high_res_scale
# tmp['y'] = (tmp['y'] - gc.loc[inst_slice + '_' + dataset, 'Y_shift']) * high_res_scale

# tmp["geometry"] = tmp.apply(
#         # swapped for some reason
#         lambda row: [row["y"], row["x"]] , axis=1
#     )

# tmp['name'] = pd.Series(tmp.index.tolist(), index=tmp.index.tolist())

# tmp[['name', 'geometry']].to_parquet('data/michal_landscape_files/E14_' + inst_slice + '/cell_metadata.parquet')

# Individual Steps

## Image

In [ ]:
# Path to your OME-TIFF file
file_path = path_data + 'registered_images/' + inst_slice + '_' + dataset_name + '.ome.tiff'

# Open the OME-TIFF file and read the image data
with tifffile.TiffFile(file_path) as tif:
    series = tif.series[0] 
    image_data = series.asarray()

In [ ]:
# image_data_scaled = image_data[:,:0] * 2
# Save the image data to a regular TIFF file without compression
tifffile.imwrite(path_landscape_files + 'output_regular.tif', image_data, compression=None)
# image_ds = dega.pre.reduce_image_size(path_landscape_files + 'output_regular.tif', image_scale, path_landscape_files)
image_png = dega.pre._convert_to_png(path_landscape_files + 'output_regular.tif')
dega.pre.make_deepzoom_pyramid(image_png, path_landscape_files + 'pyramid_images/', 'h&e', suffix=suffix)

# Spots

In [16]:
tsv_file = path_data + 'Substrate_E14_62_map_file.tsv' 

'data/IST_data/Substrate_E12_71/Substrate_E14_62_map_file.tsv'

In [29]:
# Define parameters
tsv_file = path_data + 'Substrate_E12_71_map_file.tsv' 
chunk_size = 10_000_000
parquet_prefix = path_landscape_files + 'map_parquet_files/output_chunk'

for i, chunk in enumerate(pd.read_csv(tsv_file, sep="\t", chunksize=chunk_size, header=None, index_col=0)):
    output_file = f"{parquet_prefix}_{i}.parquet"
    chunk.index.name = None
    chunk.to_parquet(output_file, engine="pyarrow")

    if i%20 == 0:
        print(f"Saved {output_file}")

print("Processing complete!")

# Region Barcodes

In [31]:
barcodes = pd.read_csv(
    path_data + 'matrix_files/T1_E12_71/T1_E12_71_raw/barcodes.tsv.gz', 
    sep='\t', 
    header=None, 
    index_col=0
)
barcodes.index.name = None
barcodes['x'] = pd.Series(index=barcodes.index.tolist())
barcodes['y'] = pd.Series(index=barcodes.index.tolist())

In [34]:
barcodes_list = barcodes.index.tolist()

for inst_file in glob(path_landscape_files + 'map_parquet_files/*.parquet'):
    
    inst_chunk = pd.read_parquet(inst_file)

    common_barcodes = list(set(inst_chunk.index.tolist()).intersection(barcodes_list))

    print(inst_file, 'found', len(common_barcodes), 'barcodes')

    if len(common_barcodes) > 0:
        barcodes.loc[common_barcodes, 'x'] = inst_chunk.loc[common_barcodes, 1]
        barcodes.loc[common_barcodes, 'y'] = inst_chunk.loc[common_barcodes, 2]
        

data/IST_landscape_files/map_parquet_files/output_chunk_50.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_40.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_2.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_32.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_22.parquet found 2344732 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_49.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_59.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_14.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_58.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_48.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_15.parquet found 0 barcodes
data/IST_landscape_files/map_parquet_files/output_chunk_41.parquet foun

In [85]:
barcodes.head()

,x,y
AAAAAAAAAAAAATACAAGCCCTAATCTGC,6991895.0,8019782.0
AAAAAAAAAAAAATAGACATGCTTAAAGAC,6132986.0,8839980.0
AAAAAAAAAAAACACCAAGCCCCGTAATAA,6791549.0,8289259.0
AAAAAAAAAAAGAGACATCAATAGCACAAA,5622690.0,8422825.0
AAAAAAAAAACACTGAGCCCTGTGCAATGC,6467943.0,8463364.0


In [35]:
barcodes.to_parquet(path_landscape_files + 'meta_spots.parquet')

## Cells

In [50]:
cells = pd.read_csv(
    path_data + '/matrix_files/' + inst_slice + '_' + dataset_name + '/' + inst_slice + '_' + dataset_name + '_cell_binned/barcodes.tsv.gz', 
    sep='\t', 
    header=None, 
    index_col=0
)

In [52]:
cells.head()

""
0
cell100000:11721:24048
cell100001:11837:24263
cell100002:12260:23447
cell100003:11841:24234
cell100004:12301:23881


In [53]:
high_res_scale = 1/0.382

In [57]:
gc = pd.read_csv(path_data + 'registered_images/globalpos_' + dataset_name + '.csv', index_col=0)
gc

,X_shift,Y_shift
Sample_ID,,
T1_E12_71,4156,6187
T2_E12_71,3665,12827
T3_E12_71,2718,20134
T4_E12_71,1959,29518
T5_E12_71,2109,37553
T6_E12_71,9420,4390
T7_E12_71,9045,11593
T8_E12_71,8631,20424
T9_E12_71,8188,28995


In [61]:
tmp = pd.DataFrame([x.split(':') for x in cells.index.tolist()])
tmp.set_index(0, inplace=True)
tmp.index.name = None
tmp.columns = ['x', 'y']
tmp = tmp.astype(float)
tmp['x'] = (tmp['x'] - gc.loc[inst_slice + '_' + dataset_name, 'X_shift']) * high_res_scale 
tmp['y'] = (tmp['y'] - gc.loc[inst_slice + '_' + dataset_name, 'Y_shift']) * high_res_scale

tmp["geometry"] = tmp.apply(
    # swapped for some reason
        lambda row: [row["y"], row["x"]] , axis=1
    )

tmp['name'] = pd.Series(tmp.index.tolist(), index=tmp.index.tolist())

tmp[['name', 'geometry']].to_parquet(path_landscape_files + 'cell_metadata.parquet')

In [62]:
print(tmp.x.min(), tmp.x.max())
print(tmp.y.min(), tmp.y.max())

2251.30890052356 11507.85340314136
3678.0104712041884 14513.0890052356


In [63]:
clusters = pd.DataFrame(index=tmp.index.tolist())
clusters['cluster'] = pd.Series(0, index=tmp.index.tolist())

In [66]:
output_dir = Path(path_landscape_files + 'cell_clusters')
output_dir.mkdir(parents=True, exist_ok=True)
clusters.to_parquet(path_landscape_files + 'cell_clusters/cluster.parquet')

## Segmented Cells

In [70]:
tile_bounds = {}
tile_bounds["x_min"] = 0
tile_bounds["x_max"] = 20000
tile_bounds["y_min"] = 0
tile_bounds["y_max"] = 20000

In [72]:
poly = pd.read_csv(path_data + 'cell_masks/' + inst_slice + '_' + dataset_name + '_Expanded_5um_cell_contour_coords.csv')

poly['vertex_x'] = (poly['vertex_x'] - gc.loc[inst_slice + '_' + dataset_name, 'Y_shift']) * high_res_scale 
poly['vertex_y'] = (poly['vertex_y'] - gc.loc[inst_slice + '_' + dataset_name, 'X_shift']) * high_res_scale 

poly.head()

,cell_id,vertex_x,vertex_y
0,1,5418.795812,2515.811518
1,1,5405.706806,2513.193717
2,1,5397.329843,2499.581152
3,1,5399.947644,2486.492147
4,1,5413.560209,2478.115183


In [73]:
# Group by 'cell_id' and aggregate the coordinates into lists
grouped = poly.groupby("cell_id").agg(list)

def safe_polygon(row):
    try:
        return Polygon(zip(row["vertex_x"], row["vertex_y"]))
    except Exception as e:
        # print(f"Error processing row {row.name}: {e}")
        return Polygon()

# # Create a new column for polygons
# grouped["geometry"] = grouped.apply(
#     lambda row: Polygon(zip(row["vertex_x"], row["vertex_y"])), axis=1
# )

grouped["geometry"] = grouped.apply(safe_polygon, axis=1)

# Convert the DataFrame with polygon data into a GeoDataFrame
cells = gpd.GeoDataFrame(grouped, geometry="geometry")[["geometry"]]


In [74]:
def simple_format(geometry, image_scale):
    # factor in scaling
    return [[[coord[0] / image_scale, coord[1] / image_scale] for coord in polygon] for polygon in geometry]

In [75]:
def transform_polygon(polygon):

    exterior_coords = polygon.exterior.coords

    # Creating the original structure by directly using numpy array for each coordinate pair
    original_format_coords = np.array([np.array(coord) for coord in exterior_coords])

    return np.array([original_format_coords], dtype=object)

In [77]:
# Apply the transformation to each polygon
cells["NEW_GEOMETRY"] = cells["geometry"].apply(
    lambda poly: transform_polygon(poly)
)

In [78]:
cells["GEOMETRY"] = cells["NEW_GEOMETRY"].apply(lambda x: simple_format(x, image_scale))

cells["polygon"] = cells["GEOMETRY"].apply(lambda x: Polygon(x[0]))

gdf_cells = gpd.GeoDataFrame(geometry=cells["polygon"])

gdf_cells["center_x"] = gdf_cells.centroid.x
gdf_cells["center_y"] = gdf_cells.centroid.y

In [79]:
gdf_cells.head()

,geometry,center_x,center_y
cell_id,,,
1,"POLYGON ((5418.796 2515.812, 5405.707 2513.194...",5415.094335,2497.528378
2,"POLYGON ((5366.440 2816.859, 5358.063 2811.099...",5358.806232,2803.762556
3,"POLYGON ((5442.356 2725.236, 5433.979 2722.094...",5444.021331,2712.722913
4,"POLYGON ((5311.466 2772.356, 5305.707 2766.597...",5311.449061,2757.712067
5,"POLYGON ((5251.257 2884.921, 5242.880 2881.780...",5250.151863,2868.427897


In [80]:
output_dir = Path(path_landscape_files + 'cell_segmentation')
output_dir.mkdir(parents=True, exist_ok=True)

In [81]:
tile_size = 250
tile_size_x = tile_size
tile_size_y = tile_size

In [82]:
# if not os.path.exists(path_output):
#     os.mkdir(path_output)

x_min = tile_bounds["x_min"]
x_max = tile_bounds["x_max"]
y_min = tile_bounds["y_min"]
y_max = tile_bounds["y_max"]

# Calculate the number of tiles needed
n_tiles_x = int(np.ceil((x_max - x_min) / tile_size_x))
n_tiles_y = int(np.ceil((y_max - y_min) / tile_size_y))
print(n_tiles_x, n_tiles_y)

80 80


In [83]:
path_output = path_landscape_files + 'cell_segmentation'

In [84]:
for i in range(n_tiles_x):

    if i % 2 == 0:
        print('row', i)

    for j in range(n_tiles_y):
        tile_x_min = x_min + i * tile_size_x
        tile_x_max = tile_x_min + tile_size_x
        tile_y_min = y_min + j * tile_size_y
        tile_y_max = tile_y_min + tile_size_y

        # find cell polygons with centroids in the tile
        keep_cells = gdf_cells[
            (gdf_cells.center_x >= tile_x_min)
            & (gdf_cells.center_x < tile_x_max)
            & (gdf_cells.center_y >= tile_y_min)
            & (gdf_cells.center_y < tile_y_max)
        ].index.tolist()

        inst_geo = cells.loc[keep_cells, ["GEOMETRY"]]

        # try adding cell name to geometry
        inst_geo["name"] = pd.Series(
            inst_geo.index.tolist(), index=inst_geo.index.tolist()
        )

        filename = f"{path_output}/cell_tile_{i}_{j}.parquet"

        # Save the filtered DataFrame to a Parquet file
        if inst_geo.shape[0] > 0:
            inst_geo[["GEOMETRY", "name"]].to_parquet(filename)

row 0
row 2
row 4
row 6
row 8
row 10
row 12
row 14
row 16
row 18
row 20
row 22
row 24
row 26
row 28
row 30
row 32
row 34
row 36
row 38
row 40
row 42
row 44
row 46
row 48
row 50
row 52
row 54
row 56
row 58
row 60
row 62
row 64
row 66
row 68
row 70
row 72
row 74
row 76
row 78


## Pseudo-Transcript Jitter

In [87]:
path_data

'data/IST_data/Substrate_E12_71/'

In [86]:
ls data/IST_data/

Substrate_E12_71/


In [88]:
dataset_name

'E12_71'

In [11]:
adata = sc.read_10x_mtx(path_data + 'matrix_files/' + inst_slice +  '_' + dataset_name + '/' + inst_slice + '_' + dataset_name + '_raw/')
adata

AnnData object with n_obs × n_vars = 10113662 × 56748
    var: 'gene_ids', 'feature_types'

In [12]:
adata_cell = sc.read_10x_mtx(path_data + 'matrix_files/' + inst_slice +  '_' + dataset_name + '/' + inst_slice + '_' + dataset_name + '_cell_binned/')
adata_cell

AnnData object with n_obs × n_vars = 134420 × 56748
    var: 'gene_ids', 'feature_types'

## Meta Gene

In [15]:
list_genes = adata.var.index.tolist()
meta_gene = pd.DataFrame(index=list_genes)
from matplotlib.colors import to_hex
# Get all categorical color palettes from Matplotlib and flatten them into a single list of colors
palettes = [plt.get_cmap(name).colors for name in plt.colormaps() if "tab" in name]
flat_colors = [color for palette in palettes for color in palette]

# Convert RGB tuples to hex codes
flat_colors_hex = [to_hex(color) for color in flat_colors]

# Use modular arithmetic to assign a color to each gene, white for genes with "Blank"
colors = [
    flat_colors_hex[i % len(flat_colors_hex)] if "Blank" not in gene else "#FFFFFF"
    for i, gene in enumerate(list_genes)
]

# Create a DataFrame with genes and their assigned colors
ser_color = pd.Series(colors, index=list_genes)

In [16]:
meta_gene['mean'] = pd.Series(100, index=list_genes)
meta_gene['std'] = pd.Series(10, index=list_genes)
meta_gene['max'] = pd.Series(100, index=list_genes)
meta_gene['non-zero'] = pd.Series(0.5, index=list_genes)
meta_gene['color'] = ser_color

In [18]:
path_landscape_files

'data/landscape_files/IST_mouse_cranium/'

In [19]:
meta_gene.to_parquet(path_landscape_files + 'meta_gene.parquet')

## Landscape Visualization 

In [92]:
server_address = dega.viz.get_local_server()

In [99]:
server_address

59698

In [16]:
landscape = dega.viz.Landscape(
    base_url = f'http://localhost:{server_address}/data/landscape_files/Xenium_V1_human_Pancreas_FFPE_outs'
)
landscape

Landscape(base_url='http://localhost:55509/data/landscape_files/Xenium_V1_human_Pancreas_FFPE_outs', cell_attr…

In [97]:
# landscape = dega.viz.Landscape(
#     base_url = f'http://localhost:{server_address}/data/IST_landscape_files'
# )

In [116]:
landscape_parameters = {
    "technology": "Xenium",
    "segmentation_approach": [
        "default"
    ],
    "max_pyramid_zoom": 16,
    "tile_size": 250,
    "image_info": [
        {
            "name": "h&e",
            "button_name": "H",
            "color": [
                0,
                0,
                255
            ]
        },
        {
            "name": "bound",
            "button_name": "BOUND",
            "color": [
                0,
                255,
                0
            ]
        },
        {
            "name": "rna",
            "button_name": "RNA",
            "color": [
                255,
                0,
                0
            ]
        },
        {
            "name": "prot",
            "button_name": "PROT",
            "color": [
                255,
                255,
                255
            ]
        }
    ],
    "image_format": ".webp",
    "use_int_index": true
}

In [10]:
### Save Landscape Parameters

In [17]:
from pathlib import Path
import json

path = Path("data/landscape_files/IST_mouse_cranium/landscape_parameters.json")

# Make sure directory exists (should be true, but this is safe)
path.parent.mkdir(parents=True, exist_ok=True)

# Write the JSON to file
with path.open("w") as f:
    json.dump(landscape_parameters, f, indent=2)


## Meta Cluster

In [28]:
pd.read_parquet('data/landscape_files/Xenium_V1_human_Pancreas_FFPE_outs/cell_clusters/meta_cluster.parquet').head()

,color,count
1,#1f77b4,17949
2,#ff7f0e,15781
3,#2ca02c,14415
4,#d62728,11840
5,#9467bd,9526


In [26]:
meta_cluster = pd.DataFrame()
meta_cluster.loc['0', 'color'] = '#ff7f0e'
meta_cluster.loc['0', 'count'] = 1000
meta_cluster.to_parquet(path_landscape_files + 'cell_clusters/meta_cluster.parquet')

In [27]:
# pd.read_parquet(path_landscape_files + 'cell_clusters/cluster.parquet')

# Viz

In [6]:
server_address = dega.viz.get_local_server()

Using placeholder transformation matrix

In [30]:
landscape_data_dir = 'data/landscape_files'
sample = 'IST_mouse_cranium'

landscape_ist = dega.viz.Landscape(
    technology='Xenium',
    base_url = f"http://localhost:{server_address}/{landscape_data_dir}/{sample}",
    height=400
)

landscape_ist

Landscape(base_url='http://localhost:55509/data/landscape_files/IST_mouse_cranium', cell_attr=['leiden'], heig…

In [20]:
# landscape_data_dir = 'data/landscape_files'
# sample = 'Xenium_V1_human_Pancreas_FFPE_outs'

# landscape_ist = dega.viz.Landscape(
#     technology='Xenium',
#     base_url = f"http://localhost:{server_address}/{landscape_data_dir}/{sample}",
#     height=400
# )

# landscape_ist